# Step 6 Backbone Refresh

This notebook reruns the clean Step 6 backbone screen.
It keeps the two models that actually mattered later: BERTweet and Twitter-RoBERTa.


## Environment Check

This first cell makes sure the notebook is attached to the right project and conda environment.


In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

EXPECTED_ENV = 'usc-nlp'
if EXPECTED_ENV not in sys.executable.lower():
    raise RuntimeError(f'This notebook should run inside the {EXPECTED_ENV} conda environment. Current executable: {sys.executable}')


def detect_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'group17_runtime.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root from the notebook location.')


def find_latest_run(root: Path, prefix: str) -> Path:
    candidates = [path for path in root.iterdir() if path.is_dir() and path.name.startswith(prefix)]
    if not candidates:
        raise FileNotFoundError(f'No run found in {root} with prefix {prefix!r}.')
    return max(candidates, key=lambda path: path.stat().st_mtime)


def read_json(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def run_command(cmd: list[str]) -> None:
    print('Running command:')
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)


PROJECT_DIR = detect_project_dir(Path.cwd().resolve())
print(f'Project directory: {PROJECT_DIR}')
print(f'Python executable: {sys.executable}')


Project directory: D:\CS544-Group17-Project
Python executable: F:\anaconda\envs\usc-nlp\python.exe


## Step 6 Plan

The key knobs live here so the overnight run stays easy to audit.


In [2]:
STEP6_SCRIPT = PROJECT_DIR / 'group17_step6_backbone_refresh.py'
OUTPUT_ROOT = PROJECT_DIR / 'step6_outputs'
RUN_ID = 'exp_s61_backbone_refresh_submission'
EXPERIMENT_IDS = [
    's61_bertweet_kwpair_anchor_v1',
    's62_twitter_roberta_kwpair_v1',
]

step6_cmd = [sys.executable, str(STEP6_SCRIPT), '--run-id', RUN_ID]
for experiment_id in EXPERIMENT_IDS:
    step6_cmd.extend(['--experiment-id', experiment_id])

pd.DataFrame({'experiment_id': EXPERIMENT_IDS})


## Launch Step 6

This run creates a fresh output folder and leaves older campaigns untouched.


In [3]:
run_command(step6_cmd)
STEP6_RUN_DIR = find_latest_run(OUTPUT_ROOT, RUN_ID)
print(f'Step 6 run directory: {STEP6_RUN_DIR}')


Running command:
F:\anaconda\envs\usc-nlp\python.exe D:\CS544-Group17-Project\group17_step6_backbone_refresh.py --run-id exp_s61_backbone_refresh_submission --experiment-id s61_bertweet_kwpair_anchor_v1 --experiment-id s62_twitter_roberta_kwpair_v1
Step 6 run directory: D:\CS544-Group17-Project\step6_outputs\exp_s61_backbone_refresh_submission_local_20260423_234211


## Campaign Summary

The campaign file is a quick way to compare both Step 6 runs side by side.


In [4]:
step6_campaign = pd.read_csv(STEP6_RUN_DIR / 'campaign_summary.csv')
display(step6_campaign)


                   experiment_id   stage                       model_name           input_name     cv_F1  cv_F1_std  cv_Precision  cv_Recall  cv_Accuracy  tuned_F1  best_threshold                                                                                                                      output_dir     status
0  s61_bertweet_kwpair_anchor_v1  screen              vinai/bertweet-base  text + keyword pair  0.810673   0.017796      0.844121   0.781274     0.845128  0.813303            0.44  D:\CS544-Group17-Project\step6_outputs\exp_s61_backbone_refresh_submission_local_20260423_234211\s61_bertweet_kwpair_anchor_v1  completed
1  s62_twitter_roberta_kwpair_v1  screen  cardiffnlp/twitter-roberta-base  text + keyword pair  0.806492   0.012941      0.832948   0.782841     0.840331  0.807617            0.59  D:\CS544-Group17-Project\step6_outputs\exp_s61_backbone_refresh_submission_local_20260423_234211\s62_twitter_roberta_kwpair_v1  completed


## Final Metrics Table

This view pulls the tuned metrics from each summary file and makes the comparison easier to quote in a report.


In [5]:
rows = []
for experiment_id in EXPERIMENT_IDS:
    summary_path = STEP6_RUN_DIR / experiment_id / f'{experiment_id}_summary.json'
    summary = read_json(summary_path)
    tuned = summary['tuned_threshold']
    rows.append({
        'experiment_id': experiment_id,
        'model_name': summary['model_name'],
        'input_name': summary['input_name'],
        'best_threshold': summary['best_threshold'],
        'F1': tuned['F1'],
        'Precision': tuned['Precision'],
        'Recall': tuned['Recall'],
        'Accuracy': tuned['Accuracy'],
    })

step6_results = pd.DataFrame(rows).sort_values('F1', ascending=False).reset_index(drop=True)
display(step6_results)


                   experiment_id                       model_name           input_name  best_threshold        F1  Precision    Recall  Accuracy
0  s61_bertweet_kwpair_anchor_v1              vinai/bertweet-base  text + keyword pair            0.44  0.813303   0.834487  0.793168  0.845129
1  s62_twitter_roberta_kwpair_v1  cardiffnlp/twitter-roberta-base  text + keyword pair            0.59  0.807617   0.847983  0.770918  0.843796
